# Multi-output model sanity notebook

Notebook to inspect a multi-output dataset and run a forward pass with a multi-output model (e.g. `mo_tbe`).

In [ ]:
from pathlib import Path
import pickle
import numpy as np
import tensorflow as tf

DATA_FILE = Path('/gpfs0/elyk/users/hovavl/GITs/TBE/multi_output/training_files.pk')
MODEL_PATH = Path('/gpfs0/elyk/users/hovavl/GITs/TBE/models/mo_tbe')

DATA_FILE, MODEL_PATH


In [ ]:
def summarize_structure(obj, name='root', depth=0, max_depth=2):
    indent = '  ' * depth
    if isinstance(obj, dict):
        print(f'{indent}{name}: dict(len={len(obj)})')
        if depth < max_depth:
            for k, v in obj.items():
                summarize_structure(v, f'[{k!r}]', depth + 1, max_depth)
    elif isinstance(obj, (list, tuple)):
        print(f'{indent}{name}: {type(obj).__name__}(len={len(obj)})')
        if depth < max_depth:
            for i, v in enumerate(obj[:5]):
                summarize_structure(v, f'[{i}]', depth + 1, max_depth)
    elif isinstance(obj, np.ndarray):
        print(f'{indent}{name}: ndarray shape={obj.shape} dtype={obj.dtype}')
    else:
        print(f'{indent}{name}: {type(obj).__name__}')

with DATA_FILE.open('rb') as f:
    data = pickle.load(f)

summarize_structure(data)


In [ ]:
def find_param_dict_and_outputs(obj):
    if isinstance(obj, (list, tuple)):
        param_dict = next((x for x in obj if isinstance(x, dict) and all(isinstance(v, np.ndarray) for v in x.values())), None)
        outputs = [x for x in obj if isinstance(x, np.ndarray)]
        if param_dict is not None and outputs:
            return param_dict, outputs
    if isinstance(obj, dict):
        dict_candidates = [v for v in obj.values() if isinstance(v, dict)]
        arr_candidates = [v for v in obj.values() if isinstance(v, np.ndarray)]
        if dict_candidates and arr_candidates:
            return dict_candidates[0], arr_candidates
    raise ValueError('Could not infer parameter dict and multi-outputs from the loaded file.')

params_dict, outputs = find_param_dict_and_outputs(data)
print('Parameter keys:', list(params_dict.keys()))
print('Number of outputs:', len(outputs))
for i, out in enumerate(outputs):
    print(f'output[{i}] shape={out.shape}')


In [ ]:
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
model.summary()

X = np.stack([params_dict[k] for k in params_dict], axis=1).astype(np.float32)
pred = model(X[:8], training=False)
pred = pred if isinstance(pred, (list, tuple)) else [pred]

print('Input batch shape:', X[:8].shape)
for i, p in enumerate(pred):
    arr = np.asarray(p)
    print(f'prediction[{i}] shape={arr.shape}')


In [ ]:
# Optional: keep sorting by redshift for now (open item: confirm preferred bin edges).
def sort_and_bin_by_redshift(redshift, *arrays, bins=(6, 7, 8, 9, 10, 11)):
    order = np.argsort(redshift)
    z_sorted = redshift[order]
    arrays_sorted = [a[order] for a in arrays]
    bin_ids = np.digitize(z_sorted, bins=bins, right=False)
    return z_sorted, arrays_sorted, bin_ids

z_key = next((k for k in params_dict if k.lower() in {'z', 'redshift', 'redshifts'}), None)
if z_key is not None:
    z_sorted, out_sorted, z_bins = sort_and_bin_by_redshift(params_dict[z_key], *outputs)
    print('Found redshift key:', z_key)
    print('sorted z[:10]:', z_sorted[:10])
    print('bin ids[:10]:', z_bins[:10])
else:
    print('No explicit redshift key found in params dict.')
